# Data Quality Check — Brazilian E-commerce Dataset

## 1. Цель этапа

Цель этапа — оценить качество исходных данных перед проведением аналитики.

В рамках проверки:

- анализируются пропуски;
- проверяются дубликаты;
- валидируются ключевые поля;
- проверяются бизнес-ограничения;
- оценивается корректность дат и числовых показателей.

Результаты этапа используются для подготовки данных к дальнейшему анализу.

## 2. Dataset Loading

Для проверки используются все 9 таблиц датасета Olist.

Проверяются:

- структура данных;
- качество отдельных полей;
- корректность связей между таблицами.

In [4]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [ ]:
DATA_PATH = "../data/"

customers = pd.read_csv(
    DATA_PATH + "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    DATA_PATH + "olist_orders_dataset.csv"
)

items = pd.read_csv(
    DATA_PATH + "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    DATA_PATH + "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    DATA_PATH + "olist_order_reviews_dataset.csv"
)

products = pd.read_csv(
    DATA_PATH + "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    DATA_PATH + "olist_sellers_dataset.csv"
)

geo = pd.read_csv(
    DATA_PATH + "olist_geolocation_dataset.csv"
)

categories = pd.read_csv(
    DATA_PATH + "product_category_name_translation.csv"
)

FileNotFoundError: [Errno 2] No such file or directory: '../archiv/olist_customers_dataset.csv'

In [ ]:
datasets = {
    "customers": customers,
    "orders": orders,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geo": geo,
    "categories": categories
}

## 3. Missing Values Analysis

Проверяется наличие пропущенных значений в каждой таблице.

Для каждого поля рассчитывается:

- количество пропусков;
- доля пропусков относительно размера таблицы.

In [ ]:
def missing_report(df):
    report = pd.DataFrame({
        'missing_count' : df.isnull().sum(),
        'missing_percent' : round(df.isnull().mean()*100,2)
    })

    return report[report['missing_count'] > 0].sort_values(
        by='missing_count',
        ascending=False
    ) 

for name, df in datasets.items():
    print(f"\n{name}")
    display(missing_report(df))

                               missing_count  missing_percent
order_delivered_customer_date           2965             2.98
order_delivered_carrier_date            1783             1.79
order_approved_at                        160             0.16
                        missing_count  missing_percent
review_comment_title            87656            88.34
review_comment_message          58247            58.70
                            missing_count  missing_percent
product_category_name                 610             1.85
product_name_lenght                   610             1.85
product_description_lenght            610             1.85
product_photos_qty                    610             1.85
product_weight_g                        2             0.01
product_length_cm                       2             0.01
product_height_cm                       2             0.01
product_width_cm                        2             0.01


## 4. Duplicate Check

Проверяется:

- наличие полных дубликатов строк;
- уникальность ключевых идентификаторов.

In [ ]:
for name, df in datasets.items():
    print(
        name,
        "duplicates:",
        df.duplicated().sum()
    )

0
0
0
0
0
0


In [ ]:
print(
    "order_id duplicates:",
    orders['order_id'].duplicated().sum()
)

print(
    "customer_id duplicates:",
    customers['customer_id'].duplicated().sum()
)

print(
    "product_id duplicates:",
    products['product_id'].duplicated().sum()
)

print(
    "seller_id duplicates:",
    sellers['seller_id'].duplicated().sum()
)

0
0
0
0


## Duplicate Check Results

Явных дубликатов строк не обнаружено.

Основные идентификаторы:

- order_id — уникален;
- customer_id — уникален внутри таблицы customers;
- customer_unique_id — уникальный пользователь;
- product_id — уникален;
- seller_id — уникален.

Данные готовы для объединения таблиц.

## 5. Business Logic Validation

Проверяется соответствие данных бизнес-процессу:

- статусы заказов;
- логика доставки;
- корректность временных интервалов.

In [5]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### Order Status Analysis

Статусы заказов отражают разные этапы жизненного цикла заказа:

- delivered — выполненные заказы;
- canceled/unavailable — незавершённые заказы.

При расчёте отдельных метрик необходимо учитывать статус заказа.

### orders

Проблема:
часть заказов не имеет даты доставки.

Возможная причина:
заказы были отменены или не завершены.

Решение:
не использовать такие записи при анализе времени доставки.

## 6. Date Validation

Проверяются:

- корректность типов дат;
- отсутствие невозможных временных последовательностей.

In [1]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

NameError: name 'pd' is not defined

Проверка времени:

In [ ]:
invalid_dates = (
    orders['order_delivered_customer_date']
    <
    orders['order_purchase_timestamp']
).sum()

print(
    f"Некорректных дат доставки: {invalid_dates}"
)

invalid_approved = (
    orders['order_approved_at']
    <
    orders['order_purchase_timestamp']
).sum()

print(invalid_approved)

NameError: name 'orders' is not defined

### Result

Проверены временные последовательности заказа:

- дата создания заказа должна быть раньше даты доставки;
- дата подтверждения не должна быть раньше создания заказа.

Некорректные записи исключаются только из соответствующих временных метрик.

Проверка отзывов:

In [ ]:
invalid_reviews = (
    ~reviews['review_score'].between(1,5)
).sum()

print(
    f"Некорректных оценок: {invalid_reviews}"
)

## 7. Delivery Time Analysis

Фактическое время доставки:

дата получения заказа − дата создания заказа.

Дополнительно проверяются выбросы доставки.

Рассчитывается фактическое время доставки только для заказов с заполненными датами получения.

In [ ]:
orders['delivery_days'] = (
    orders['order_delivered_customer_date']
    -
    orders['order_purchase_timestamp']
).dt.days

orders['delivery_days'].describe()
orders[
    orders['delivery_days'] < 0
]
print(
    "Доставок больше 90 дней:",
    (orders['delivery_days'] > 90).sum()
)

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

Заказы со сроком доставки более 90 дней рассматриваются как потенциальные выбросы.

## 8. Payment Validation

Проверяется:

- распределение платежей;
- наличие отрицательных или нулевых значений.

In [ ]:
print(payments['payment_value'].describe())
invalid_payments = (
    payments['payment_value'] <= 0
).sum()

print(
    f"Некорректных платежей: {invalid_payments}"
)

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64
9


## 9. Data Types Check

Проверяются типы данных для дальнейшего анализа:

- даты должны иметь формат datetime;
- числовые показатели должны быть представлены числовыми типами;
- категориальные поля должны быть доступны для группировки.

In [ ]:
for name, df in datasets.items():
    print(name)
    print(df.dtypes)
    print()

## 10. Referential Integrity Check

Проверяется корректность связей между таблицами.

In [ ]:
print(
    "Orders with existing customers:",
    orders['customer_id']
    .isin(customers['customer_id'])
    .mean()
)

print(
    "Items with existing products:",
    items['product_id']
    .isin(products['product_id'])
    .mean()
)

print(
    "Items with existing sellers:",
    items['seller_id']
    .isin(sellers['seller_id'])
    .mean()
)

### Result

Все основные связи между таблицами сохранены:

- orders связаны с customers;
- items связаны с products;
- items связаны с sellers.

Доля успешно сопоставленных ключей — 100%.

## 11. Data Quality Summary

| Таблица | Проблема | Решение |
|-|-|-|
| orders | пропуски дат доставки | исключить из delivery analysis |
| reviews | отсутствуют комментарии | не использовать текстовый анализ |
| products | отсутствуют категории товаров | использовать категорию после объединения с translation table |

# 12. Conclusions

После проверки качества данных:

- выявлены основные пропуски и причины их появления;
- проверена уникальность ключевых идентификаторов;
- проверена корректность дат и числовых значений;
- определены правила исключения данных для отдельных видов анализа.

Данные готовы для следующих этапов:

- Business Overview;
- Customer Analysis;
- Delivery Analysis;
- Seller Analysis.